# Data preparation


In [ ]:
import os
import io
import json
import math
import glob
import random
import hashlib
import tarfile
import shutil
import urllib.request
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import soundfile as sf
import scipy.signal as sps

In [ ]:
DATA_ROOT = Path("data")
RAW_DIR = DATA_ROOT / "raw"
WORK_DIR = DATA_ROOT / "work"
MANIFEST_DIR = DATA_ROOT / "manifests"
SAMPLE_RATE = 16000
CHUNK_SECONDS = 30
N_SAMPLES = SAMPLE_RATE * CHUNK_SECONDS
for d in (RAW_DIR, WORK_DIR, MANIFEST_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
SOURCES = {
    "librispeech-train-clean-100": {
        "url": "https://www.openslr.org/resources/12/train-clean-100.tar.gz",
        "language": "en",
        "task": "transcribe",
    },
    "librispeech-train-clean-360": {
        "url": "https://www.openslr.org/resources/12/train-clean-360.tar.gz",
        "language": "en",
        "task": "transcribe",
    },
    "librispeech-train-other-500": {
        "url": "https://www.openslr.org/resources/12/train-other-500.tar.gz",
        "language": "en",
        "task": "transcribe",
    },
    "mls-spanish": {
        "url": "https://dl.fbaipublicfiles.com/mls/mls_spanish_opus.tar.gz",
        "language": "es",
        "task": "transcribe",
    },
    "mls-german": {
        "url": "https://dl.fbaipublicfiles.com/mls/mls_german_opus.tar.gz",
        "language": "de",
        "task": "transcribe",
    },
    "mls-french": {
        "url": "https://dl.fbaipublicfiles.com/mls/mls_french_opus.tar.gz",
        "language": "fr",
        "task": "transcribe",
    },
    "covost2-es-en": {
        "url": "https://dl.fbaipublicfiles.com/covost/covost_v2.es_en.tsv.tar.gz",
        "language": "es",
        "task": "translate",
    },
    "covost2-de-en": {
        "url": "https://dl.fbaipublicfiles.com/covost/covost_v2.de_en.tsv.tar.gz",
        "language": "de",
        "task": "translate",
    },
}

def fetch(name):
    spec = SOURCES[name]
    dest = RAW_DIR / (name + ".tar.gz")
    if dest.exists():
        return dest
    tmp = dest.with_suffix(".part")
    with urllib.request.urlopen(spec["url"]) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f, length=1 << 20)
    tmp.rename(dest)
    return dest

def extract(name):
    outdir = RAW_DIR / name
    if outdir.exists():
        return outdir
    with tarfile.open(fetch(name)) as tf:
        tf.extractall(outdir)
    return outdir

In [ ]:
@dataclass
class Utterance:
    utt_id: str
    audio_path: str
    text: str
    language: str
    task: str
    duration: float
    source: str
    speaker: str = ""
    translation: str = ""

def sha1_16(s):
    return hashlib.sha1(s.encode()).hexdigest()[:16]

def scan_librispeech(root, source, language):
    utts = []
    for trans in sorted(root.rglob("*.trans.txt")):
        chapter_dir = trans.parent
        with open(trans) as f:
            for line in f:
                utt_id, text = line.strip().split(" ", 1)
                flac = chapter_dir / (utt_id + ".flac")
                if not flac.exists():
                    continue
                info = sf.info(str(flac))
                utts.append(Utterance(
                    utt_id=sha1_16(source + utt_id),
                    audio_path=str(flac),
                    text=text.lower(),
                    language=language,
                    task="transcribe",
                    duration=info.frames / info.samplerate,
                    source=source,
                    speaker=utt_id.split("-")[0],
                ))
    return utts

def scan_mls(root, source, language):
    utts = []
    for split in ("train",):
        trans = root / "train" / "transcripts.txt"
        if not trans.exists():
            continue
        audio_root = root / "train" / "audio"
        with open(trans) as f:
            for line in f:
                utt_id, text = line.strip().split("\t", 1)
                spk, book, _ = utt_id.split("_")
                opus = audio_root / spk / book / (utt_id + ".opus")
                if not opus.exists():
                    continue
                info = sf.info(str(opus))
                utts.append(Utterance(
                    utt_id=sha1_16(source + utt_id),
                    audio_path=str(opus),
                    text=text,
                    language=language,
                    task="transcribe",
                    duration=info.frames / info.samplerate,
                    source=source,
                    speaker=spk,
                ))
    return utts

def scan_covost(root, source, language):
    utts = []
    for tsv in root.rglob("*.tsv"):
        df = pd.read_csv(tsv, sep="\t", quoting=3, on_bad_lines="skip")
        if "translation" not in df.columns:
            continue
        for row in df.itertuples():
            path = root / "clips" / str(row.path)
            if not path.exists():
                continue
            info = sf.info(str(path))
            utts.append(Utterance(
                utt_id=sha1_16(source + str(row.path)),
                audio_path=str(path),
                text=str(row.sentence),
                language=language,
                task="translate",
                duration=info.frames / info.samplerate,
                source=source,
                speaker=str(getattr(row, "client_id", "")),
                translation=str(row.translation),
            ))
    return utts

In [ ]:
SCANNERS = {
    "librispeech-train-clean-100": scan_librispeech,
    "librispeech-train-clean-360": scan_librispeech,
    "librispeech-train-other-500": scan_librispeech,
    "mls-spanish": scan_mls,
    "mls-german": scan_mls,
    "mls-french": scan_mls,
    "covost2-es-en": scan_covost,
    "covost2-de-en": scan_covost,
}

def build_manifest(name):
    spec = SOURCES[name]
    root = extract(name)
    utts = SCANNERS[name](root, name, spec["language"])
    out = MANIFEST_DIR / (name + ".jsonl")
    with open(out, "w") as f:
        for u in utts:
            f.write(json.dumps(u.__dict__, ensure_ascii=False) + "\n")
    return out, len(utts)

manifests = {}
for name in SOURCES:
    try:
        path, n = build_manifest(name)
        manifests[name] = (path, n)
        print(name, n)
    except FileNotFoundError as e:
        print(name, "skipped", e)

In [ ]:
MIN_DURATION = 0.6
MAX_DURATION = 30.0
MIN_CHARS = 2
MAX_CHARS_PER_SECOND = 30.0

def load_manifest(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def duration_filter(rows):
    kept = []
    for r in rows:
        if r["duration"] < MIN_DURATION or r["duration"] > MAX_DURATION:
            continue
        if len(r["text"]) < MIN_CHARS:
            continue
        if len(r["text"]) / max(r["duration"], 1e-6) > MAX_CHARS_PER_SECOND:
            continue
        kept.append(r)
    return kept

def alignment_proxy_filter(rows):
    kept = []
    for r in rows:
        words = r["text"].split()
        if not words:
            continue
        wps = len(words) / max(r["duration"], 1e-6)
        if wps < 0.3 or wps > 7.5:
            continue
        longest = max(len(w) for w in words)
        if longest > 40:
            continue
        kept.append(r)
    return kept

def dedupe(rows):
    seen = set()
    kept = []
    for r in rows:
        key = (r["language"], r["task"], r["text"][:200])
        if key in seen:
            continue
        seen.add(key)
        kept.append(r)
    return kept

In [ ]:
import re

APOSTROPHES = {"\u2019": "'", "\u02bc": "'", "\u2018": "'"}
DASHES = {"\u2013": "-", "\u2014": "-", "\u2212": "-"}
QUOTES = {"\u201c": '"', "\u201d": '"', "\u00ab": '"', "\u00bb": '"'}

def clean_text(s):
    for table in (APOSTROPHES, DASHES, QUOTES):
        for k, v in table.items():
            s = s.replace(k, v)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def machine_text_score(s):
    if not s:
        return 1.0
    letters = sum(c.isalpha() for c in s)
    digits = sum(c.isdigit() for c in s)
    upper = sum(c.isupper() for c in s)
    ratio_upper = upper / max(letters, 1)
    ratio_digits = digits / max(len(s), 1)
    score = 0.0
    if ratio_upper > 0.6 and letters > 10:
        score += 0.5
    if ratio_digits > 0.4:
        score += 0.5
    if "http" in s or "www." in s:
        score += 1.0
    return score

def text_filter(rows):
    kept = []
    for r in rows:
        t = clean_text(r["text"])
        if not t:
            continue
        if machine_text_score(t) >= 0.5:
            continue
        r = dict(r)
        r["text"] = t
        if r.get("translation"):
            r["translation"] = clean_text(r["translation"])
        kept.append(r)
    return kept

In [ ]:
filtered = {}
for name, (path, _) in manifests.items():
    rows = load_manifest(path)
    n0 = len(rows)
    rows = duration_filter(rows)
    rows = alignment_proxy_filter(rows)
    rows = text_filter(rows)
    rows = dedupe(rows)
    filtered[name] = rows
    print(name, n0, "->", len(rows))

In [ ]:
def resample_to_16k(wave, sr):
    if sr == SAMPLE_RATE:
        return wave.astype(np.float32)
    g = math.gcd(sr, SAMPLE_RATE)
    return sps.resample_poly(wave, SAMPLE_RATE // g, sr // g).astype(np.float32)

def load_wave_16k(path):
    wave, sr = sf.read(path, dtype="float32", always_2d=True)
    wave = wave.mean(axis=1)
    return resample_to_16k(wave, sr)

def peak_normalize(wave, target=0.95):
    peak = np.abs(wave).max()
    if peak < 1e-8:
        return wave
    return wave * min(1.0, target / peak)

def trim_silence(wave, frame=400, hop=160, threshold_db=-45.0):
    if len(wave) < frame:
        return wave
    frames = np.lib.stride_tricks.sliding_window_view(wave, frame)[::hop]
    rms = np.sqrt((frames ** 2).mean(axis=1) + 1e-12)
    db = 20 * np.log10(rms + 1e-12)
    voiced = np.where(db > threshold_db)[0]
    if len(voiced) == 0:
        return wave
    start = max(voiced[0] * hop - hop * 4, 0)
    end = min((voiced[-1] + 1) * hop + frame + hop * 4, len(wave))
    return wave[start:end]

In [ ]:
SHARD_UTTS = 2048

def write_shards(rows, split_name):
    outdir = WORK_DIR / split_name
    outdir.mkdir(parents=True, exist_ok=True)
    shard_idx = 0
    buf_audio = []
    buf_meta = []
    def flush():
        nonlocal shard_idx, buf_audio, buf_meta
        if not buf_meta:
            return
        shard = outdir / f"shard-{shard_idx:05d}"
        np.savez_compressed(str(shard) + ".npz", *buf_audio)
        with open(str(shard) + ".jsonl", "w") as f:
            for m in buf_meta:
                f.write(json.dumps(m, ensure_ascii=False) + "\n")
        shard_idx += 1
        buf_audio = []
        buf_meta = []
    for r in rows:
        try:
            wave = load_wave_16k(r["audio_path"])
        except Exception:
            continue
        wave = peak_normalize(trim_silence(wave))
        if len(wave) < int(MIN_DURATION * SAMPLE_RATE):
            continue
        meta = dict(r)
        meta["samples"] = int(len(wave))
        buf_audio.append(wave)
        buf_meta.append(meta)
        if len(buf_meta) >= SHARD_UTTS:
            flush()
    flush()
    return shard_idx

In [ ]:
rng = random.Random(20260301)
all_rows = []
for name, rows in filtered.items():
    all_rows.extend(rows)
rng.shuffle(all_rows)

by_speaker = {}
for r in all_rows:
    by_speaker.setdefault((r["source"], r["speaker"]), []).append(r)

speakers = list(by_speaker)
rng.shuffle(speakers)
n_dev = max(len(speakers) // 100, 1)
dev_speakers = set(speakers[:n_dev])
test_speakers = set(speakers[n_dev:2 * n_dev])

train_rows, dev_rows, test_rows = [], [], []
for key, rows in by_speaker.items():
    if key in dev_speakers:
        dev_rows.extend(rows)
    elif key in test_speakers:
        test_rows.extend(rows)
    else:
        train_rows.extend(rows)
print(len(train_rows), len(dev_rows), len(test_rows))

In [ ]:
for split, rows in (("train", train_rows), ("dev", dev_rows), ("test", test_rows)):
    n = write_shards(rows, split)
    print(split, n, "shards")

In [ ]:
def split_stats(rows):
    hours = sum(r["duration"] for r in rows) / 3600
    langs = {}
    tasks = {}
    for r in rows:
        langs[r["language"]] = langs.get(r["language"], 0) + r["duration"]
        tasks[r["task"]] = tasks.get(r["task"], 0) + r["duration"]
    return {
        "utterances": len(rows),
        "hours": round(hours, 1),
        "language_hours": {k: round(v / 3600, 1) for k, v in sorted(langs.items())},
        "task_hours": {k: round(v / 3600, 1) for k, v in sorted(tasks.items())},
    }

report = {
    "train": split_stats(train_rows),
    "dev": split_stats(dev_rows),
    "test": split_stats(test_rows),
}
with open(MANIFEST_DIR / "split-report.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))